# Day I - B) 2D Magnetostatics

- **<font color='green'>[RUN & OBSERVE]</font>** $\rightarrow$ run directly
- **<font color='orange'>[RUN & PLAY]</font>** $\rightarrow$ run directly, change parameters
- **<font color='red'>[FILL & RUN]</font>**    $\rightarrow$ fill before running
- **<font color='magenta'>[FILL & PLAY]</font>** $\rightarrow$ fill and change parameters


---
## Outlines:

<a id='outlines'></a>

- [**1) Theoretical preliminaries**](#1_theory) (*5 minutes*)
- [**2) Geometry**](#2_geometry) (*5 minutes*)
- [**3) Material properties**](#3_material_properties) (*5 minutes*)
- [**4) Function space + residual**](#4_function_space_residual) (*10 minutes*)
    - [a) Function space](#4a_fes)
    - [b) Residual functional](#4b_residual)
- [**5) Solver framework**](#5_solver_framework) (*30 minutes*)
    - [a) Fixed-point](#5a_fixed_point)
    - [b) Kač́anov / Picard](#5b_kacanov) 
    - [c) Newton](#5c_newton)
    - [d) Comparison](#5d_comparison) 
- [**6) More elaborated Newton**](#6_more_elaborated_newton) (*5 minutes*)

|**<u>Objective</u>** : Implement and compare fixed-point, Kačanov and Newton nonlinear solvers|
|---|
|*Indicative duration* : 1 hour|

---
## 1) Theoretical preliminaries

|*Indicative duration* : 5 minutes|
|---|

<a id='1_theory'></a>


This notebook is based on [this paper](https://arxiv.org/abs/2403.18520) presented at SCEE 2024 by Egger, Radu and Engertsberger, which present a unified framework to solve non-linear magnetostatic problems.


We want to find $a \in H^1_0(\Omega)$ such that the residual vanishes:

$$ R(a; v) = \int_\Omega \nu(|\text{curl}\,a|^2)\,\text{curl}\,v\cdot\text{curl}\,a \;-\; \int_\Omega v\,j \;=\; 0 \qquad \forall v\in H^1_0(\Omega) $$

Starting from an initial guess $a^0$, we construct a sequence $a^n$ via the **update formula**:

$$ a^{n+1} = a^n + \tau^n\,\delta a^n, \qquad n \geq 0 $$

where $\delta a^n$ is the **search direction**, defined by solving the linear variational problem:

$$ \int_\Omega \nu^n\,\text{curl}\,\delta a^n \cdot \text{curl}\,v  \;=\; -R(a^n; v) \qquad \forall v\in H^1_0(\Omega) $$

The RHS $-R(a^n; v)$ expands to:
$$ -R(a^n; v) = \int_\Omega v\,j \;-\; \int_\Omega \nu(|\text{curl}\,a^n|^2)\,\text{curl}\,v\cdot\text{curl}\,a^n $$

We can prove that we can find $\tau$ such that the method converges to the solution.

The three solvers differ **only** in the choice of the generalized reluctivity $\nu^n$:

| Solver | Choice of $\nu^n$ | Matrix | Factorization |
|---|---|---|---|
| **Fixed-point** | $\nu^n = \bar\nu > 0$ (constant, damped by $\omega$) | **Same** every step | **Once** before loop |
| **Kačanov** | $\nu^n = \nu(\lVert \text{curl} \,a^n\rVert^2)$ (chord, frozen at $a^n$) | Changes each step | Each iteration |
| **Newton** | $\nu^n = \partial_b^2 w(\text{curl}\,a^n)$ (Jacobian) | Changes each step | Each iteration |


<a id='2_geometry'></a>
_______
## 2) Geometry

|*Indicative duration* : 5 minutes|
|---|

We define a typical inductance geometry: an iron core with winding, no airgap (so there is a risk of saturation!).

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from utils.myGeometries import gaped_inductor
mesh = gaped_inductor(airgap = 0, h = 0.1)  # no airgap

from ngsolve.webgui import Draw
print(f"Regions : {mesh.GetMaterials()}")
print(f"Boundaries : {mesh.GetBoundaries()}")
Draw(mesh.MaterialCF({"condN":1, "iron":2, "condP":3}), mesh)

### Linear Magnetostatics

We want to solve the following weak formulation, i.e., find $a\in H^1_0 (\Omega)$ such that

$$ \forall v\in H^1_0(\Omega) , \quad \int_\Omega \text{curl}~ v  \cdot \nu \text{curl}~ a  = \int_\Omega v j $$

- in our 2D problem: $\text{curl}(\cdot) =  \begin{pmatrix} 0 & 1 \\-1 & 0 \end{pmatrix} \text{grad}(\cdot)$

- $\nu$ is the magnetic reluctivity :
    - $\nu = 1/\mu_0 $ in air and conductors
    - $\nu = 1/(1000 \mu_0) $ in iron; it might even depend on $B = \text{curl}~ a$

- $j$ is the current density :
    - $ j = 0 $ in air and iron
    - $ j= +J $ in positive conductor
    - $ j = -J$ in negative conductor
  
with $J = 10\;A/mm^2$. 

|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
# Parameters
from numpy import pi
mu0 = 4e-7 * pi   # H/m, void magnetic permeability
mur = 1000        # relative permeability of iron
J = 10e6          # A/m²

# Formulation
from ngsolve import H1, BilinearForm, LinearForm, dx, grad, CoefficientFunction
fes = H1(mesh, order = 1, dirichlet = "out")

R = CoefficientFunction(((0, 1), (-1, 0)), dims = (2, 2))  # -pi/2 rotation
curl = lambda u: R * grad(u)

a, v = fes.TnT()
A = BilinearForm(fes)
A += 1/mu0 * curl(a) * curl(v) * dx("air|condP|condN")
A += 1/(mur*mu0) * curl(a) * curl(v) * dx("iron")
l = LinearForm(fes)
l += v * J * dx("condP") 
l += - v * J * dx("condN")

# Assembly
K = A.Assemble().mat
f = l.Assemble().vec

# Resolution
from ngsolve import GridFunction
sol = GridFunction(fes)
sol.vec.data = K.Inverse(freedofs = fes.FreeDofs()) * f
Draw(sol)

<a id='3_material_properties'></a>
_______
## 3) Material properties

|*Indicative duration* : 5 minutes|
|---|

The magnetic reluctivity in iron depends on $b^2 = |\text{curl}\,a|^2$:

$$ \nu_{\text{iron}}(b^2) = 100 + 10\exp(1.8\,b^2) $$

Its derivative w.r.t. $b^2$:

$$ \nu'(b^2) = \frac{d\nu}{db^2}(b^2) = 18\exp(1.8\,b^2) $$

**<font color='red'> Q1. Is the reluctivity model realistic? </font>**

|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
from numpy import pi
from ngsolve import exp
import numpy as np
import matplotlib.pyplot as plt

mu0 = 4e-7 * pi      # H/m, void magnetic permeability
J = 10e6              # A/m^2, current density

def nu_iron(b2):
    """ magnetic reluctivity of iron w.r.t b2"""  
    return 100 + 10 * exp(1.8 * b2)

def dnu_iron_db2(b2):
    """ derivative of the magnetic reluctivity of iron w.r.t b2 """  
    return 18 * exp(1.8 * b2)

# Plot BH curve
b = np.linspace(0, 2, 100)
H_vals = nu_iron(b**2) * b   # H = nu(B²)*B  (scalar field)
plt.figure(figsize=(6, 4))
plt.plot(H_vals, b)
plt.xlabel("H (A/m)"); plt.ylabel("B (T)")
plt.title("BH curve of iron (nonlinear)")
plt.grid(True); plt.tight_layout()
plt.show()

<a id='4_function_space_residual'></a>
_______
## 4) Function space + residual

|*Indicative duration* : 10 minutes|
|---|

<a id='4a_fes'></a>
### a) Function space

First we define the function space.

**NB**: we are in 2D, so no need for $H(\text{curl})$. The curl operator can be written
$$ \text{curl}(\cdot) \;=\; \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix} \text{grad}(\cdot) $$

|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
from ngsolve import H1
fes = H1(mesh, order = 1, dirichlet = "out")

from ngsolve import grad, dx, LinearForm, BilinearForm, CoefficientFunction, GridFunction


<a id='4b_residual'></a>
### b) Residual functional

The residual $R(a; v)$ reads:

$$ R(a; v) = \int_\Omega \nu(|\text{curl}\,a|^2)\,\text{curl}\,a\cdot\text{curl}\,v - \int_\Omega v\,j $$

We recall the NGSolve syntax:
- `a += u * v * dx("coilP")` will integrate the form on `coilP` domain
- `a += u * v * dx("coilP|coilN")` will integrate the form on the union of `coilP` and `coilN` domains


|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
a, v = fes.TnT()

def residual(a, v):
    """ Nonlinear residual functional. """
    b2 = curl(a)**2  # |curl(a)|^2 (scalar)
    res = curl(v) * (1/mu0) * curl(a) * dx("air|condP|condN")
    res += curl(v) * nu_iron(b2) * curl(a) * dx("iron")
    res += -v * J * dx("condP") + v * J * dx("condN")
    return res

<a id='5_solver_framework'></a>

--- 
## 5) Solver framework

|*Indicative duration* : 15 minutes|
|---|

At each iteration $n$:
1. Compute residual vector $\mathbf{r} = R(a^n, \phi_i)$
2. Solve $K(\nu^n)\,\mathbf{d} = \mathbf{r} \Rightarrow$ we have to define the linear operator $K(\nu^n)$
3. Update: $\mathbf{a}^{n+1} = \mathbf{a}^n - \tau^n \mathbf{d}$ (since $\delta a = -\mathbf{d}$)
4. Backtracking: halve $\tau$ if residual does not decrease

The difference between each method is the linear operator.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from utils.mySolvers import iterative_solve

help(iterative_solve)

solutions = {}
residuals = {}
times = {}

### a) Fixed-point: $\nu^n = \bar\nu$ (constant)
<a id='5a_fixed_point'></a>

|*Indicative duration* : 5 minutes|
|---|

The linear operator reads
$$ L(\delta a^n, v) = \int_\Omega \bar\nu \, \text{curl}\,\delta a^n \cdot \text{curl}\,v$$

The matrix is **iterate-independent**, so we factorize it **once** before the loop. Each iteration is then just a cheap sparse solve on triangular matrices.

When it converges with suitable step size, it converges **linearly**.


|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
# FIXED-POINT: nu^n = bar_nu (constant, iterate-independent)
bar_nu_iron = 10000.0   # constant reluctivity in iron (hyperparameter)
fp_relax = 0.1         # fixed relaxation (damping); must stay below stability limit

def lin_op_fixed_point(da, v, aOld):
    """bar_nu = constant chi. Matrix is the same every iteration."""
    b_da_v = curl(da) * curl(v)
    bf = b_da_v * (1/mu0) * dx("air|condP|condN")
    bf += b_da_v * bar_nu_iron * dx("iron")
    return bf

# Execute fixed point
sol_fp, res_fp, t_fp = iterative_solve( fes, residual, 
                                       lin_op_fixed_point,
                                       factor_once=True, 
                                       constant_step=fp_relax,
                                       verbosity=2)
solutions["fixed_point"] = sol_fp
residuals["fixed_point"] = res_fp
times["fixed_point"] = t_fp

# Draw result
B = curl(sol_fp)
print("Fixed point flux density")
Draw(B, mesh,
     vectors={"grid_size": 50, "offset": 0.5},
     settings={"Objects": {"Wireframe": False, "Surface": False}})

<a id='5b_kacanov'></a>
### b) Kačanov / Picard: $\nu^n = \nu(|\text{curl}\,a^n|^2)$ (chord reluctivity)
|*Indicative duration* : 5 minutes|
|---|

The linear operator reads
$$ L(\delta a^n, v) = \int_\Omega \nu(|\text{curl}\,a^n|^2) \, \text{curl}\,\delta a^n \cdot \text{curl}\,v$$

This recovers the classical **"frozen coefficient"** (or Picard) iteration.

It corresponds to the intuition that we solve the magnetostatic problem with a initial $B$ that is zero; so the permeability and next $B$ is over-evaluated. We evaluate the permeability again, it has a smaller value, we evaluate $B$ again that will be smaller, and we loop.



|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
def lin_op_kacanov(da, v, aOld):
    """Chord reluctivity: nu(|curl(aOld)|^2). Matrix changes each iteration."""
    b2 = curl(aOld)**2
    b_da_v = curl(da) * curl(v)
    bf = b_da_v * (1/mu0) * dx("air|condP|condN")
    bf += b_da_v * nu_iron(b2) * dx("iron")
    return bf

# Execute Kačanov / Picard
sol_kc, res_kc, t_kc = iterative_solve(fes, residual, 
                                       lin_op_kacanov,
                                       verbosity=2)

solutions["kacanov"] = sol_kc
residuals["kacanov"] = res_kc
times["kacanov"] = t_kc

# Draw result
B = curl(sol_fp)
print("Fixed point flux density")
Draw(B, mesh,
     vectors={"grid_size": 50, "offset": 0.5},
     settings={"Objects": {"Wireframe": False, "Surface": False}})

<a id='5c_newton'></a>
### c) Newton: $\nu^n = \partial_b^2 w(\text{curl}\,a^n)$ (full Jacobian)
|*Indicative duration* : 10 minutes|
|---|

The linear operator reads
$$ L(\delta a^n, v) = \int_\Omega \partial_b^2 w(\text{curl}\,a^n) \, \text{curl}\,\delta a^n \cdot \text{curl}\,v$$

This is the **full tangent** operator.

**<font color='red'>Q1. Derive the explicit expression of the Newton linear operator.</font>**



|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
def lin_op_newton(da, v, aOld):
    """Full Newton Jacobian. Quadratic convergence; matrix changes each iter."""
    b2 = curl(aOld)**2
    b_da_v = curl(da) * curl(v)
    b_a_v = curl(aOld) * curl(v)
    b_a_da = curl(aOld) * curl(da)
    bf = b_da_v * (1/mu0) * dx("air|condP|condN")
    bf += b_da_v * nu_iron(b2) * dx("iron")
    bf += 2 * b_a_v * b_a_da * dnu_iron_db2(b2) * dx("iron")
    return bf

sol_nt, res_nt, t_nt = iterative_solve(fes, residual,
                                       lin_op_newton,
                                       verbosity=2)

# Execute Newton
solutions["newton"] = sol_nt
residuals["newton"] = res_nt
times["newton"] = t_nt

# Draw result
B = curl(sol_fp)
print("Fixed point flux density")
Draw(B, mesh,
     vectors={"grid_size": 50, "offset": 0.5},
     settings={"Objects": {"Wireframe": False, "Surface": False}})

<a id='5d_comparison'></a>

### d) Comparison

|*Indicative duration* : 10 minutes|
|---|

**<font color='red'>Q2. Change the parameters (mesh, step size, relaxation factors...) and observe the behavior of the methos.</font>**

|**<font color='orange'>[RUN & PLAY]</font>**|
|---|

In [ ]:
print("\n=== Wall-time comparison ===")
for key in solutions:
    niter = len(residuals[key])
    wall = times[key] * 1000
    print(f"  {key:12s}: {niter:3d} iters, {wall:8.1f} ms")

for key in residuals:
    plt.semilogy(residuals[key], marker='o', markersize=4, linewidth=1.5, label=key)
plt.xlabel("Iteration number"); plt.ylabel(r"Residual norm")
plt.title("Convergence: fixed-point vs Kačanov/Picard vs Newton")
plt.grid(True, which="both"); plt.legend()
plt.tight_layout()
plt.show()

---
## 6) More elaborated Newton
<a id='6_more_elaborated_newton'></a>

|*Indicative duration* : 5 minutes|
|---|

We can improve by : 
- considering more stopping criteria
- adding proper linesearch based on Armijo's condition
- using NGSolve automatic differentiation capabilities

You can analyze the algorithm in the file `utils\mySolver`, and make some comments.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from utils.mySolvers import solveNL

result = solveNL(fes, residual, verbosity = 3)

plt.semilogy(result["residual"])
plt.grid()
plt.xlabel("Iterations"); plt.ylabel("Norm of the residual");
plt.show()

---
## 7) Reference

- H. Egger, F. Engertsberger, B. Radu, Global convergence of iterative solvers for problems of nonlinear magnetostatics, march 2024, [https://arxiv.org/abs/2403.18520](https://arxiv.org/abs/2403.18520)

[*Back to top - outlines*](#outlines)